⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 1,
  "metadata": {
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Migration: `insert_trg_loc.txt`\n",
        "\n",
        "**Conversion Timestamp:** 2024-07-30 12:00:00\n",
        "\n",
        "This notebook contains the conversion of an ODI SQL insert statement for the `TRG_LOC` table."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "# SCEN_TASK_NO {10}: Setup ETL Parameters (standard widgets)\n",
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"ETL Job Type\");\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"-1\", \"Datasource Number ID\");\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"-1\", \"ETL Process WID\");\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"-1\", \"ODI Session Number\");\n",
        "dbutils.widgets.text(\"ETL_LAST_EXTRACT_TIME\", \"1900-01-01 00:00:00\", \"ETL Last Extract Time\");\n",
        "dbutils.widgets.text(\"ETL_CURRENT_EXTRACT_TIME\", \"1900-01-01 00:00:00\", \"ETL Current Extract Time\");"
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "# ETL Parameters\n",
        "\n",
        "Displaying the values of the initialized ETL parameters."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "display(spark.sql(\"\"\"\n",
        "  SELECT\n",
        "    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,\n",
        "    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,\n",
        "    '${ETL_PROC_WID}' AS ETL_PROC_WID,\n",
        "    '${ODI_SESS_NO}' AS ODI_SESS_NO,\n",
        "    '${ETL_LAST_EXTRACT_TIME}' AS ETL_LAST_EXTRACT_TIME,\n",
        "    '${ETL_CURRENT_EXTRACT_TIME}' AS ETL_CURRENT_EXTRACT_TIME\n",
        "\"\"\"))"
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Target Table DDL\n",
        "\n",
        "Ensuring the target table `workspace.hr.trg_loc` exists with the appropriate schema."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {20}: (No specific DDL in original for this task, inferred for target table)\n",
        "CREATE TABLE IF NOT EXISTS workspace.hr.trg_loc (\n",
        "    LOCATION_ID      BIGINT,\n",
        "    STREET_ADDRESS   STRING,\n",
        "    POSTAL_CODE      STRING,\
        "    CITY             STRING,\n",
        "    STATE_PROVINCE   STRING,\n",
        "    COUNTRY_ID       STRING\n",
        ")\n",
        "USING DELTA;\n",
        ""
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Data Load\n",
        "\n",
        "Loading data into `workspace.hr.trg_loc` from `workspace.hr.locations`."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {30}: Insert into HR.TRG_LOC\n",
        "INSERT INTO workspace.hr.trg_loc\n",
        "  (\n",
        "    LOCATION_ID ,\n",
        "    STREET_ADDRESS ,\n",
        "    POSTAL_CODE ,\n",
        "    CITY ,\n",
        "    STATE_PROVINCE ,\n",
        "    COUNTRY_ID\n",
        "  )\n",
        "SELECT\n",
        "  locations.LOCATION_ID ,\n",
        "  locations.STREET_ADDRESS ,\n",
        "  locations.POSTAL_CODE ,\n",
        "  locations.CITY ,\n",
        "  locations.STATE_PROVINCE ,\n",
        "  locations.COUNTRY_ID\n",
        "FROM\n",
        "  workspace.hr.locations AS locations;\n",
        ""
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Validation\n",
        "\n",
        "Checking the record count and sampling the loaded data."
      ]
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) FROM workspace.hr.trg_loc;"
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "code",
      "source": [
        "-- MAGIC %sql\n",
        "SELECT * FROM workspace.hr.trg_loc LIMIT 10;"
      ],
      "metadata": {
        "collapsed": false
      },
      "execution_count": null,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Cleanup\n",
        "\n",
        "No temporary tables were created for this specific direct insert, so no cleanup is required here."
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Conversion Notes\n",
        "\n",
        "1.  **Schema Conversion**: `HR` schema converted to `workspace.hr`.\n",
        "2.  **Table Naming**: `TRG_LOC` and `LOCATIONS` converted to lowercase `trg_loc` and `locations`.\n",
        "3.  **Oracle Hints Removed**: `/*+ APPEND PARALLEL */` hint was removed as it's Oracle-specific and not applicable to Databricks Delta Lake.\n",
        "4.  **Inferred DDL**: The `CREATE TABLE IF NOT EXISTS` statement for `workspace.hr.trg_loc` was inferred based on standard column types typically found in HR `LOCATIONS` tables, as the original ODI snippet did not provide DDL.\n",
        "5.  **SCEN_TASK_NO**: The `SCEN_TASK_NO` comments from the ODI source have been preserved as comments within the respective SQL cells."
      ]
    }
  ]
}